In [ ]:
# Import libraries
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load the data

train_df = pd.read_csv(
    train_file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

test_df = pd.read_csv(
    test_file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Training data:")
print(train_df.head())

print("\nValidation data:")
print(test_df.head())

print("\nTraining shape:", train_df.shape)
print("Validation shape:", test_df.shape)

print("\nLabel distribution:")
print(train_df["label"].value_counts())

In [ ]:
# Convert ham/spam into 0/1

train_df["label"] = train_df["label"].map({
    "ham": 0,
    "spam": 1
})

test_df["label"] = test_df["label"].map({
    "ham": 0,
    "spam": 1
})

X_train = train_df["message"].astype(str).values
y_train = train_df["label"].values

X_test = test_df["message"].astype(str).values
y_test = test_df["label"].values

print(X_train[:5])
print(y_train[:5])

In [ ]:
# Convert text into integer sequences

from tensorflow.keras.layers import TextVectorization

max_tokens = 10000
sequence_length = 100

vectorizer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Learn vocabulary from training messages
vectorizer.adapt(X_train)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))

In [ ]:
# Build the SMS classification model

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

model = Sequential([
    vectorizer,

    Embedding(
        input_dim=max_tokens,
        output_dim=64
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Build the SMS classification model

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

model = Sequential([
    vectorizer,

    Embedding(
        input_dim=max_tokens,
        output_dim=64
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Train the model

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=8,
    batch_size=32,
    verbose=1
)

In [ ]:
# Evaluate model on validation data

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

In [ ]:
# Plot accuracy

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
def predict_message(pred_text):

    # Convert the message into a TensorFlow string tensor
    text_tensor = tf.convert_to_tensor(
        [pred_text],
        dtype=tf.string
    )

    # Make prediction
    prediction_probability = model(text_tensor, training=False)

    # Get the probability value
    prediction_probability = float(
        prediction_probability.numpy()[0][0]
    )

    # Convert probability to ham/spam
    if prediction_probability >= 0.5:
        label = "spam"
    else:
        label = "ham"

    return [prediction_probability, label]


pred_text = "how are you doing today?"

prediction = predict_message(pred_text)

print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
